<a href="https://colab.research.google.com/github/heyiamheshan/ASR-Nemotron-3.5-ASR-NVIDIA/blob/main/Copy_of_Qwen_Sinhala_ASR_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q --upgrade \
    "torch" "torchvision" "torchaudio" \
    "transformers==4.57.6" \
    "accelerate==1.12.0" \
    "qwen-asr==0.0.6" \
    "soundfile" "librosa" "numpy"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 122.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.6/141.6 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.8/416.8 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 871.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.6 MB/s eta 0:00

In [ ]:
import torch, torchvision, transformers
print("torch:       ", torch.__version__)
print("torchvision: ", torchvision.__version__)
print("transformers:", transformers.__version__)
print("CUDA:        ", torch.cuda.is_available())

# This is the import that failed last time
from transformers import AutoProcessor
print("✅ AutoProcessor imports fine")

import qwen_asr
print("✅ qwen_asr imports fine")

torch:        2.14.1+cu130
torchvision:  0.29.1+cu130
transformers: 4.57.6
CUDA:         True
✅ AutoProcessor imports fine
✅ qwen_asr imports fine


In [ ]:
import torch
from qwen_asr import Qwen3ASRModel

device = "cuda"

model = Qwen3ASRModel(
    model="Qwen/Qwen3-ASR-0.6B",
    device=device,
)

print("✅ Model loaded on", device)

TypeError: Qwen3ASRModel.__init__() got an unexpected keyword argument 'device'

In [ ]:
import inspect
from qwen_asr import Qwen3ASRModel

print(inspect.signature(Qwen3ASRModel.__init__))
print()
print(Qwen3ASRModel.__init__.__doc__)

(self, backend: str, model: Any, processor: Any, sampling_params: Optional[Any] = None, forced_aligner: Optional[qwen_asr.inference.qwen3_forced_aligner.Qwen3ForcedAligner] = None, max_inference_batch_size: int = -1, max_new_tokens: int = 512)

None


In [ ]:
import qwen_asr
import inspect

# What's exported at the top level?
print("Top-level names:")
print([n for n in dir(qwen_asr) if not n.startswith("_")])
print()

# Look for a from_pretrained / load helper
for name in dir(qwen_asr):
    if name.startswith("_"):
        continue
    obj = getattr(qwen_asr, name)
    if callable(obj) and any(k in name.lower() for k in ("load", "pretrain", "build", "init", "auto")):
        print(f"--- {name} ---")
        try:
            print(inspect.signature(obj))
        except Exception as e:
            print("(no signature)", e)
        print()

# Also check for a classmethod on the model itself
print("Qwen3ASRModel methods:")
print([m for m in dir(qwen_asr.Qwen3ASRModel) if not m.startswith("_")])

Top-level names:
['Qwen3ASRModel', 'Qwen3ForcedAligner', 'core', 'inference', 'parse_asr_output']

Qwen3ASRModel methods:
['LLM', 'finish_streaming_transcribe', 'from_pretrained', 'get_supported_languages', 'init_streaming_state', 'streaming_transcribe', 'transcribe']


In [ ]:
import inspect
from qwen_asr import Qwen3ASRModel

print("from_pretrained:")
print(inspect.signature(Qwen3ASRModel.from_pretrained))
print()
print("transcribe:")
print(inspect.signature(Qwen3ASRModel.transcribe))
print()
print("streaming_transcribe:")
print(inspect.signature(Qwen3ASRModel.streaming_transcribe))

from_pretrained:
(pretrained_model_name_or_path: str, forced_aligner: Optional[str] = None, forced_aligner_kwargs: Optional[Dict[str, Any]] = None, max_inference_batch_size: int = 32, max_new_tokens: Optional[int] = 512, **kwargs) -> 'Qwen3ASRModel'

transcribe:
(self, audio: Union[str, Tuple[numpy.ndarray, int], List[Union[str, Tuple[numpy.ndarray, int]]]], context: Union[str, List[str]] = '', language: Union[str, List[Optional[str]], NoneType] = None, return_time_stamps: bool = False) -> List[qwen_asr.inference.qwen3_asr.ASRTranscription]

streaming_transcribe:
(self, pcm16k: numpy.ndarray, state: qwen_asr.inference.qwen3_asr.ASRStreamingState) -> qwen_asr.inference.qwen3_asr.ASRStreamingState


In [ ]:
import torch
from qwen_asr import Qwen3ASRModel

model = Qwen3ASRModel.from_pretrained(
    "Qwen/Qwen3-ASR-0.6B",
    max_inference_batch_size=1,
)

print("✅ Model loaded")
print("Supported languages:", model.get_supported_languages())

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

✅ Model loaded
Supported languages: ['Chinese', 'English', 'Cantonese', 'Arabic', 'German', 'French', 'Spanish', 'Portuguese', 'Indonesian', 'Italian', 'Korean', 'Russian', 'Thai', 'Vietnamese', 'Japanese', 'Turkish', 'Hindi', 'Malay', 'Dutch', 'Swedish', 'Danish', 'Finnish', 'Polish', 'Czech', 'Filipino', 'Persian', 'Greek', 'Romanian', 'Hungarian', 'Macedonian']


In [ ]:
langs = model.get_supported_languages()

print(f"Total languages: {len(langs)}")
print()
print(langs)
print()

# The key question
sinhala = [l for l in langs if "sinhal" in l.lower() or "si" == l.lower()]
print("Sinhala match:", sinhala if sinhala else "❌ NOT in the list")

Total languages: 30

['Chinese', 'English', 'Cantonese', 'Arabic', 'German', 'French', 'Spanish', 'Portuguese', 'Indonesian', 'Italian', 'Korean', 'Russian', 'Thai', 'Vietnamese', 'Japanese', 'Turkish', 'Hindi', 'Malay', 'Dutch', 'Swedish', 'Danish', 'Finnish', 'Polish', 'Czech', 'Filipino', 'Persian', 'Greek', 'Romanian', 'Hungarian', 'Macedonian']

Sinhala match: ❌ NOT in the list


In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files("Nerdstorm/Qwen3-ASR-0.6B-Sinhala-8bit")
for f in sorted(files):
    print(f)

.gitattributes
LICENSE-APACHE-2.0
README.md
chat_template.json
config.json
generation_config.json
merges.txt
model.safetensors
model.safetensors.index.json
preprocessor_config.json
tokenizer.json
tokenizer_config.json
vocab.json


In [ ]:
from huggingface_hub import hf_hub_download
import json

cfg_path = hf_hub_download("Nerdstorm/Qwen3-ASR-0.6B-Sinhala-8bit", "config.json")
cfg = json.load(open(cfg_path))

print("quantization block:")
print(json.dumps(cfg.get("quantization", "none found"), indent=2))
print()
print("top-level keys:", list(cfg.keys()))

config.json: 0.00B [00:00, ?B/s]

quantization block:
{
  "group_size": 64,
  "bits": 8,
  "mode": "affine"
}

top-level keys: ['architectures', 'model_type', 'quantization', 'quantization_config', 'support_languages', 'thinker_config', 'transformers_version']


In [ ]:
print("Architectures:", cfg["architectures"])
print()
langs = cfg.get("support_languages")
print(f"Supported languages ({len(langs) if langs else 0}):")
print(langs)
print()
sin = [l for l in (langs or []) if "sinhal" in str(l).lower()]
print("Sinhala present:", sin if sin else "❌ not found")
print()
print("quantization_config:", json.dumps(cfg.get("quantization_config"), indent=2))

Architectures: ['Qwen3ASRForConditionalGeneration']

Supported languages (31):
['Chinese', 'English', 'Cantonese', 'Arabic', 'German', 'French', 'Spanish', 'Portuguese', 'Indonesian', 'Italian', 'Korean', 'Russian', 'Thai', 'Vietnamese', 'Japanese', 'Turkish', 'Hindi', 'Malay', 'Dutch', 'Swedish', 'Danish', 'Finnish', 'Polish', 'Czech', 'Filipino', 'Persian', 'Greek', 'Romanian', 'Hungarian', 'Macedonian', 'Sinhala']

Sinhala present: ['Sinhala']

quantization_config: {
  "group_size": 64,
  "bits": 8,
  "mode": "affine"
}


In [ ]:
from huggingface_hub import snapshot_download

src = snapshot_download(
    repo_id="Nerdstorm/Qwen3-ASR-0.6B-Sinhala-8bit",
    local_dir="./sinhala-mlx",
)
print("✅ Downloaded to:", src)

import os
for f in sorted(os.listdir(src)):
    size = os.path.getsize(os.path.join(src, f)) / 1e6
    print(f"  {f:35s} {size:8.1f} MB")

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.01G [00:00<?, ?B/s]

LICENSE-APACHE-2.0: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/330 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Downloaded to: /content/sinhala-mlx
  .cache                                   0.0 MB
  .gitattributes                           0.0 MB
  LICENSE-APACHE-2.0                       0.0 MB
  README.md                                0.0 MB
  chat_template.json                       0.0 MB
  config.json                              0.0 MB
  generation_config.json                   0.0 MB
  merges.txt                               1.7 MB
  model.safetensors                     1006.2 MB
  model.safetensors.index.json             0.1 MB
  preprocessor_config.json                 0.0 MB
  tokenizer.json                           4.8 MB
  tokenizer_config.json                    0.0 MB
  vocab.json                               2.8 MB


In [ ]:
from safetensors import safe_open

path = "./sinhala-mlx/model.safetensors"

with safe_open(path, framework="pt") as f:
    keys = list(f.keys())

print(f"Total tensors: {len(keys)}\n")

scales = [k for k in keys if k.endswith(".scales")]
biases = [k for k in keys if k.endswith(".biases")]
print(f"Quantized layers: {len(scales)} scales, {len(biases)} biases\n")

# Look at one quantized layer in full
if scales:
    stem = scales[0][:-len(".scales")]
    print(f"Example layer: {stem}")
    with safe_open(path, framework="pt") as f:
        for suffix in ("weight", "scales", "biases"):
            k = f"{stem}.{suffix}"
            if k in keys:
                t = f.get_slice(k)
                print(f"  {suffix:8s} shape={t.get_shape()}  dtype={t.get_dtype()}")

print("\nFirst 10 tensor names:")
for k in keys[:10]:
    print(" ", k)

Total tensors: 1005

Quantized layers: 197 scales, 197 biases

Example layer: model.embed_tokens
  weight   shape=[151936, 256]  dtype=U32
  scales   shape=[151936, 16]  dtype=BF16
  biases   shape=[151936, 16]  dtype=BF16

First 10 tensor names:
  audio_tower.conv2d1.bias
  audio_tower.conv2d1.weight
  audio_tower.conv2d2.bias
  audio_tower.conv2d2.weight
  audio_tower.conv2d3.bias
  audio_tower.conv2d3.weight
  audio_tower.conv_out.weight
  audio_tower.layers.0.fc1.bias
  audio_tower.layers.0.fc1.weight
  audio_tower.layers.0.fc2.bias


In [ ]:
import torch, json, os
from safetensors import safe_open
from safetensors.torch import save_file

SRC = "./sinhala-mlx/model.safetensors"
OUT_DIR = "./sinhala-fp16"
os.makedirs(OUT_DIR, exist_ok=True)

GROUP_SIZE, BITS = 64, 8
PACK = 32 // BITS          # 4 int8 values per uint32

def dequant(w_u32, scales, biases):
    # unpack 4 x uint8 from each uint32 (little-endian order)
    w = w_u32.to(torch.int64)
    shifts = torch.arange(PACK, device=w.device) * BITS
    unpacked = (w.unsqueeze(-1) >> shifts) & 0xFF      # [..., cols, 4]
    unpacked = unpacked.reshape(*w.shape[:-1], -1)      # [..., cols*4]

    # group-wise affine dequant: x = q * scale + bias
    rows, cols = unpacked.shape
    g = unpacked.reshape(rows, cols // GROUP_SIZE, GROUP_SIZE).float()
    s = scales.float().unsqueeze(-1)
    b = biases.float().unsqueeze(-1)
    return (g * s + b).reshape(rows, cols).to(torch.float16)

with safe_open(SRC, framework="pt") as f:
    keys = set(f.keys())
    stems = {k[:-7] for k in keys if k.endswith(".scales")}

    out = {}
    for i, k in enumerate(sorted(keys)):
        if k.endswith((".scales", ".biases")):
            continue
        stem = k[:-7] if k.endswith(".weight") else None
        if stem in stems:
            out[k] = dequant(
                f.get_tensor(k),
                f.get_tensor(f"{stem}.scales"),
                f.get_tensor(f"{stem}.biases"),
            )
        else:
            t = f.get_tensor(k)
            out[k] = t.to(torch.float16) if t.is_floating_point() else t

        if (i + 1) % 200 == 0:
            print(f"  processed {i+1}/{len(keys)}")

print(f"\nDequantized {len(stems)} layers, {len(out)} tensors total")
save_file(out, f"{OUT_DIR}/model.safetensors", metadata={"format": "pt"})
print("✅ Saved:", os.path.getsize(f"{OUT_DIR}/model.safetensors") / 1e9, "GB")

  processed 200/1005

Dequantized 197 layers, 611 tensors total
✅ Saved: 1.56492096 GB


In [ ]:
import shutil, json, os

SRC_DIR = "./sinhala-mlx"
OUT_DIR = "./sinhala-fp16"

for fn in ["generation_config.json", "preprocessor_config.json",
           "tokenizer.json", "tokenizer_config.json",
           "vocab.json", "merges.txt", "chat_template.json"]:
    src = os.path.join(SRC_DIR, fn)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(OUT_DIR, fn))
        print("copied", fn)

# config minus the quantization blocks
cfg = json.load(open(os.path.join(SRC_DIR, "config.json")))
cfg.pop("quantization", None)
cfg.pop("quantization_config", None)
cfg["torch_dtype"] = "float16"

json.dump(cfg, open(os.path.join(OUT_DIR, "config.json"), "w"), indent=2)
print("\n✅ config.json written (quantization removed)")
print("Sinhala in config:", "Sinhala" in cfg.get("support_languages", []))
print("\nFinal contents:")
for f in sorted(os.listdir(OUT_DIR)):
    print(f"  {f:32s} {os.path.getsize(os.path.join(OUT_DIR,f))/1e6:8.1f} MB")

copied generation_config.json
copied preprocessor_config.json
copied tokenizer.json
copied tokenizer_config.json
copied vocab.json
copied merges.txt
copied chat_template.json

✅ config.json written (quantization removed)
Sinhala in config: True

Final contents:
  chat_template.json                    0.0 MB
  config.json                           0.0 MB
  generation_config.json                0.0 MB
  merges.txt                            1.7 MB
  model.safetensors                  1564.9 MB
  preprocessor_config.json              0.0 MB
  tokenizer.json                        4.8 MB
  tokenizer_config.json                 0.0 MB
  vocab.json                            2.8 MB


In [ ]:
import torch
from qwen_asr import Qwen3ASRModel

model = Qwen3ASRModel.from_pretrained(
    "./sinhala-fp16",
    max_inference_batch_size=1,
)

print("✅ Loaded")
langs = model.get_supported_languages()
print(f"Languages: {len(langs)}")
print("Sinhala supported:", "Sinhala" in langs)


Some weights of the model checkpoint at ./sinhala-fp16 were not used when initializing Qwen3ASRForConditionalGeneration: ['audio_tower.conv2d1.bias', 'audio_tower.conv2d1.weight', 'audio_tower.conv2d2.bias', 'audio_tower.conv2d2.weight', 'audio_tower.conv2d3.bias', 'audio_tower.conv2d3.weight', 'audio_tower.conv_out.weight', 'audio_tower.layers.0.fc1.bias', 'audio_tower.layers.0.fc1.weight', 'audio_tower.layers.0.fc2.bias', 'audio_tower.layers.0.fc2.weight', 'audio_tower.layers.0.final_layer_norm.bias', 'audio_tower.layers.0.final_layer_norm.weight', 'audio_tower.layers.0.self_attn.k_proj.bias', 'audio_tower.layers.0.self_attn.k_proj.weight', 'audio_tower.layers.0.self_attn.out_proj.bias', 'audio_tower.layers.0.self_attn.out_proj.weight', 'audio_tower.layers.0.self_attn.q_proj.bias', 'audio_tower.layers.0.self_attn.q_proj.weight', 'audio_tower.layers.0.self_attn.v_proj.bias', 'audio_tower.layers.0.self_attn.v_proj.weight', 'audio_tower.layers.0.self_attn_layer_norm.bias', 'audio_tower.

✅ Loaded
Languages: 30
Sinhala supported: False


In [ ]:
import torch, os
from safetensors import safe_open
from safetensors.torch import save_file

SRC = "./sinhala-fp16/model.safetensors"

def remap(k):
    if k.startswith("audio_tower."):
        return "thinker." + k
    if k.startswith("layers."):
        return "thinker.model." + k
    if k in ("embed_tokens.weight", "norm.weight"):
        return "thinker.model." + k
    if k == "lm_head.weight":
        return "thinker.lm_head.weight"
    return "thinker." + k        # fallback

out = {}
with safe_open(SRC, framework="pt") as f:
    for k in f.keys():
        out[remap(k)] = f.get_tensor(k)

# lm_head is tied to embed_tokens in this model — add it if absent
if "thinker.lm_head.weight" not in out:
    out["thinker.lm_head.weight"] = out["thinker.model.embed_tokens.weight"].clone()
    print("added tied lm_head")

save_file(out, SRC, metadata={"format": "pt"})
print(f"✅ Remapped {len(out)} tensors")
print("\nSample keys:")
for k in list(out)[:3]:
    print(" ", k)

added tied lm_head
✅ Remapped 612 tensors

Sample keys:
  thinker.audio_tower.conv2d1.bias
  thinker.audio_tower.conv2d1.weight
  thinker.audio_tower.conv2d2.bias


In [ ]:
import torch
from qwen_asr import Qwen3ASRModel

model = Qwen3ASRModel.from_pretrained(
    "./sinhala-fp16",
    max_inference_batch_size=1,
)

print("✅ Loaded")
langs = model.get_supported_languages()
print(f"Languages: {len(langs)}")
print("Sinhala supported:", "Sinhala" in langs)

RuntimeError: Error(s) in loading state_dict for Conv2d:
	size mismatch for weight: copying a param with shape torch.Size([480, 3, 3, 1]) from checkpoint, the shape in current model is torch.Size([480, 1, 3, 3]).

In [ ]:
import torch
from safetensors import safe_open
from safetensors.torch import save_file

SRC = "./sinhala-fp16/model.safetensors"

out = {}
fixed = []
with safe_open(SRC, framework="pt") as f:
    for k in f.keys():
        t = f.get_tensor(k)
        # MLX conv2d: [out, H, W, in]  →  PyTorch: [out, in, H, W]
        if "conv2d" in k and k.endswith(".weight") and t.ndim == 4:
            t = t.permute(0, 3, 1, 2).contiguous()
            fixed.append((k, tuple(t.shape)))
        out[k] = t

save_file(out, SRC, metadata={"format": "pt"})

print(f"✅ Permuted {len(fixed)} conv2d weights")
for k, s in fixed:
    print(f"  {k} → {s}")

✅ Permuted 3 conv2d weights
  thinker.audio_tower.conv2d1.weight → (480, 1, 3, 3)
  thinker.audio_tower.conv2d2.weight → (480, 480, 3, 3)
  thinker.audio_tower.conv2d3.weight → (480, 480, 3, 3)


In [ ]:
import torch
from qwen_asr import Qwen3ASRModel

model = Qwen3ASRModel.from_pretrained(
    "./sinhala-fp16",
    max_inference_batch_size=1,
)

print("✅ Loaded")
langs = model.get_supported_languages()
print(f"Languages: {len(langs)}")
print("Sinhala supported:", "Sinhala" in langs)

✅ Loaded
Languages: 30
Sinhala supported: False


In [ ]:
import json

# What the config on disk says
cfg = json.load(open("./sinhala-fp16/config.json"))
print("config support_languages:", len(cfg["support_languages"]))
print("Sinhala in config file:", "Sinhala" in cfg["support_languages"])
print()

# What the loaded model object carries
mc = model.model.config
print("loaded config has support_languages:", hasattr(mc, "support_languages"))
if hasattr(mc, "support_languages"):
    print("  count:", len(mc.support_languages))
    print("  Sinhala:", "Sinhala" in mc.support_languages)
print()

# Where get_supported_languages actually reads from
import inspect
print(inspect.getsource(type(model).get_supported_languages))

config support_languages: 31
Sinhala in config file: True

loaded config has support_languages: True
  count: 31
  Sinhala: True

    def get_supported_languages(self) -> List[str]:
        """
        Returns the supported language list.

        Returns:
            List[str]: Canonical language names.
        """
        return list(SUPPORTED_LANGUAGES)



In [ ]:
from IPython.display import Javascript, display
from google.colab import output
from base64 import b64decode
import io, soundfile as sf, numpy as np

RECORD = """
const sleep = ms => new Promise(r => setTimeout(r, ms));
var record = time => new Promise(async resolve => {
  const stream = await navigator.mediaDevices.getUserMedia({audio:true});
  const rec = new MediaRecorder(stream);
  const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  rec.start();
  await sleep(time);
  rec.onstop = async () => {
    const blob = new Blob(chunks);
    const buf = await blob.arrayBuffer();
    const b64 = btoa(String.fromCharCode(...new Uint8Array(buf)));
    resolve('data:audio/webm;base64,' + b64);
  };
  rec.stop();
  stream.getTracks().forEach(t => t.stop());
});
"""

def record(sec=6):
    display(Javascript(RECORD))
    print(f"🎤 Recording {sec}s — speak Sinhala now...")
    s = output.eval_js(f"record({sec*1000})")
    b = b64decode(s.split(",")[1])
    open("rec.webm", "wb").write(b)
    !ffmpeg -y -loglevel error -i rec.webm -ar 16000 -ac 1 rec.wav
    print("✅ Saved rec.wav")
    return "rec.wav"

path = record(6)


<IPython.core.display.Javascript object>

🎤 Recording 6s — speak Sinhala now...


KeyboardInterrupt: 

In [ ]:
from google.colab import files
import subprocess, os

print("Select a Sinhala audio file (wav/mp3/m4a/ogg):")
up = files.upload()

src = list(up.keys())[0]
subprocess.run(["ffmpeg", "-y", "-loglevel", "error",
                "-i", src, "-ar", "16000", "-ac", "1", "rec.wav"], check=True)

print("✅ Converted to rec.wav")
print("   size:", os.path.getsize("rec.wav") / 1000, "KB")

Select a Sinhala audio file (wav/mp3/m4a/ogg):


KeyboardInterrupt: 

In [ ]:
import subprocess, os

# OpenSLR 52 — Sinhala read speech (the dataset this model was trained on)
url = "https://www.openslr.org/resources/52/asr_sinhala_0.zip"

print("Downloading sample (this is ~1 GB, may take a few minutes)...")
subprocess.run(["wget", "-q", "--show-progress", url, "-O", "sinhala_sample.zip"], check=True)
subprocess.run(["unzip", "-q", "-o", "sinhala_sample.zip", "-d", "slr52"], check=True)

# Find the audio files
for root, dirs, fs in os.walk("slr52"):
    wavs = [f for f in fs if f.endswith((".wav", ".flac"))]
    if wavs:
        print(f"\n{root}: {len(wavs)} audio files")
        print("  examples:", wavs[:3])
        break


slr52/asr_sinhala/data/00: 736 audio files
  examples: ['00bf1ea871.flac', '004863b209.flac', '0089a9ccd9.flac']


In [ ]:
import os, subprocess, glob

# Find the transcript file
for root, dirs, fs in os.walk("slr52"):
    for f in fs:
        if f.endswith((".tsv", ".txt", ".csv")) and "utt" in f.lower() or f == "utt_spk_text.tsv":
            print("transcripts:", os.path.join(root, f))

# List what's at the top level
print("\nContents of slr52/asr_sinhala:")
for f in sorted(os.listdir("slr52/asr_sinhala")):
    p = os.path.join("slr52/asr_sinhala", f)
    kind = "dir" if os.path.isdir(p) else f"{os.path.getsize(p)/1000:.0f} KB"
    print(f"  {f:30s} {kind}")

transcripts: slr52/asr_sinhala/utt_spk_text.tsv

Contents of slr52/asr_sinhala:
  LICENSE                        20 KB
  data                           dir
  utt_spk_text.tsv               16154 KB


In [ ]:
import csv, os, subprocess, glob

# Load transcript map: utterance_id → text
trans = {}
with open("slr52/asr_sinhala/utt_spk_text.tsv", encoding="utf-8") as f:
    for row in csv.reader(f, delimiter="\t"):
        if len(row) >= 3:
            trans[row[0]] = row[2]

print(f"Loaded {len(trans)} transcripts\n")

# Pick a clip we have audio for
flacs = sorted(glob.glob("slr52/asr_sinhala/data/00/*.flac"))
utt_id = os.path.basename(flacs[0]).replace(".flac", "")

print("Test clip:", utt_id)
print("Ground truth:", trans.get(utt_id, "⚠️ not found"))

# Convert to 16kHz mono wav
subprocess.run(["ffmpeg", "-y", "-loglevel", "error",
                "-i", flacs[0], "-ar", "16000", "-ac", "1", "rec.wav"], check=True)
print("\n✅ rec.wav ready")


Error: field larger than field limit (131072)

In [ ]:
import os, subprocess, glob

trans = {}
with open("slr52/asr_sinhala/utt_spk_text.tsv", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) >= 3:
            trans[parts[0]] = parts[2]

print(f"Loaded {len(trans)} transcripts\n")

flacs = sorted(glob.glob("slr52/asr_sinhala/data/00/*.flac"))
utt_id = os.path.basename(flacs[0]).replace(".flac", "")

print("Test clip:   ", utt_id)
print("Ground truth:", trans.get(utt_id, "⚠️ not found"))

subprocess.run(["ffmpeg", "-y", "-loglevel", "error",
                "-i", flacs[0], "-ar", "16000", "-ac", "1", "rec.wav"], check=True)
print("\n✅ rec.wav ready")


Loaded 185293 transcripts

Test clip:    0000f47c22
Ground truth: මහවැලි ගඟට ගොස් ආපසු එන ගමනේදී

✅ rec.wav ready


In [ ]:
results = model.transcribe("rec.wav", language="Sinhala")
r = results[0]

print("Ground truth:", trans[utt_id])
print("Model output:", r.text)
print()
print("Raw result object:", r)

ValueError: Unsupported language: Sinhala. Supported: ['Chinese', 'English', 'Cantonese', 'Arabic', 'German', 'French', 'Spanish', 'Portuguese', 'Indonesian', 'Italian', 'Korean', 'Russian', 'Thai', 'Vietnamese', 'Japanese', 'Turkish', 'Hindi', 'Malay', 'Dutch', 'Swedish', 'Danish', 'Finnish', 'Polish', 'Czech', 'Filipino', 'Persian', 'Greek', 'Romanian', 'Hungarian', 'Macedonian']

In [ ]:
results = model.transcribe("rec.wav")   # no language arg
r = results[0]

print("Ground truth:", trans[utt_id])
print("Model output:", r.text)
print()
print("Full result:", r)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Ground truth: මහවැලි ගඟට ගොස් ආපසු එන ගමනේදී
Model output: මහවැලි ගඟට ගොස් ආපසු එන්න ගමනේදී

Full result: ASRTranscription(language='Sinhala', text='මහවැලි ගඟට ගොස් ආපසු එන්න ගමනේදී', time_stamps=None)


In [ ]:
import glob, os, subprocess

flacs = sorted(glob.glob("slr52/asr_sinhala/data/00/*.flac"))[:10]

total_cer, n = 0, 0
for fp in flacs:
    uid = os.path.basename(fp).replace(".flac", "")
    if uid not in trans:
        continue
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error",
                    "-i", fp, "-ar", "16000", "-ac", "1", "t.wav"], check=True)
    hyp = model.transcribe("t.wav")[0].text
    ref = trans[uid]

    # simple char error rate
    import difflib
    sm = difflib.SequenceMatcher(None, ref, hyp)
    cer = 1 - sm.ratio()
    total_cer += cer; n += 1

    print(f"[{n}] CER {cer:.1%}")
    print(f"    ref: {ref}")
    print(f"    hyp: {hyp}\n")

print(f"=== Average CER over {n} clips: {total_cer/n:.2%} ===")

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[1] CER 3.2%
    ref: මහවැලි ගඟට ගොස් ආපසු එන ගමනේදී
    hyp: මහවැලි ගඟට ගොස් ආපසු එන්න ගමනේදී



Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[2] CER 0.0%
    ref: උන්වහන්සේ කපාපු
    hyp: උන්වහන්සේ කපාපු



Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


...

[7] CER 7.7%
    ref: ඊට අවසරයද හිමිවූ බව ඇය කියන්නීය.
    hyp: ඊට අවසරය ද හිමි වූ බව ඇය කියන්නේය



Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[8] CER 5.7%
    ref: යම්කිසි මනුෂ්‍යයෙක් සිල් රකිනවානම්
    hyp: යම්කිසි මනුෂ්‍යයෙක් සියලු රකිනවා නම්



Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[9] CER 0.0%
    ref: ඒක නිසාම මම
    hyp: ඒක නිසාම මම

[10] CER 2.0%
    ref: අපි පෘථිවියට ඉහලින් අභ්‍යවකාශයේ රඳවා තිබෙන උපකරණයක්.
    hyp: අපි පෘථිවියට ඉහලින් අභ්‍යවකාශය රඳවා තිබෙන උපකරණයක්

=== Average CER over 10 clips: 2.63% ===


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import shutil

shutil.make_archive("/content/sinhala-asr-fp16", "zip", "./sinhala-fp16")

import os
mb = os.path.getsize("/content/sinhala-asr-fp16.zip") / 1e6
print(f"✅ Zipped: {mb:.0f} MB")

KeyboardInterrupt: 